In [ ]:
###########################################################

In [ ]:
from groq import Groq
client = Groq()

print(client.models.list())

In [ ]:
#Production-Stable RAG Assistant

In [8]:

# ============================================================
# STEP 1: INSTALL REQUIRED PACKAGES
# ============================================================

!pip install -q -U \
langchain \
langchain-community \
langchain-core \
langchain-text-splitters \
langchain-huggingface \
langchain-groq \
faiss-cpu \
pypdf \
fastapi \
uvicorn \
nest-asyncio \
pyngrok \
python-multipart \
sentence-transformers

print("✅ Packages installed successfully")


# ============================================================
# STEP 2: IMPORTS
# ============================================================

import os
import shutil
import threading
import asyncio

from pathlib import Path
from getpass import getpass

import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI, UploadFile, File
from fastapi.responses import HTMLResponse
from pydantic import BaseModel

import uvicorn

from pyngrok import ngrok

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_huggingface import (
    HuggingFaceEmbeddings
)

from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate

from langchain_core.output_parsers import (
    StrOutputParser
)

from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableLambda
)

from langchain_groq import ChatGroq

print("✅ Imports successful")


# ============================================================
# STEP 3: GROQ API KEY
# ============================================================

# Get API key:
# https://console.groq.com/keys

if "GROQ_API_KEY" not in os.environ:

    os.environ["GROQ_API_KEY"] = getpass(
        "🔑 Enter your GROQ API Key: "
    )

print("✅ GROQ API Key Loaded")


# ============================================================
# STEP 4: RAG ENGINE
# ============================================================

UPLOAD_DIR = Path("uploaded_pdfs")
UPLOAD_DIR.mkdir(exist_ok=True)


# ============================================================
# EMBEDDING MODEL
# ============================================================

EMBED_MODEL = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)


# ============================================================
# GROQ LLM
# ============================================================

LLM = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0
)


vectorstore = None
retriever = None
rag_chain = None


# ============================================================
# PROMPT TEMPLATE
# ============================================================

RAG_PROMPT = PromptTemplate.from_template("""

You are a helpful AI assistant.

Answer ONLY from the provided context.

If the answer is not found in the context,
say:
"I couldn't find this in the document."

Keep answers accurate and concise.

Context:
{context}

Question:
{question}

Answer:

""")


# ============================================================
# HELPER FUNCTIONS
# ============================================================


def format_docs(docs):

    return "\n\n".join(
        doc.page_content for doc in docs
    )



def ingest_pdf(pdf_path: str):

    global vectorstore
    global retriever
    global rag_chain

    # Load PDF
    docs = PyPDFLoader(pdf_path).load()

    # Split PDF into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150
    )

    chunks = splitter.split_documents(docs)

    # Create FAISS Vector Store
    vectorstore = FAISS.from_documents(
        chunks,
        EMBED_MODEL
    )

    # Retriever
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 5}
    )

    # RAG Chain
    rag_chain = (
        {
            "context":
                retriever
                | RunnableLambda(format_docs),

            "question":
                RunnablePassthrough()
        }

        | RAG_PROMPT
        | LLM
        | StrOutputParser()
    )

    return {
        "pages": len(docs),
        "chunks": len(chunks),
        "message": "✅ PDF ingested successfully!"
    }



def ask_question(query: str):

    global rag_chain
    global retriever

    if rag_chain is None:

        return {
            "error": "No document loaded."
        }

    try:

        answer = rag_chain.invoke(query)

        source_docs = retriever.invoke(query)

        sources = []

        for d in source_docs:

            sources.append({

                "page":
                    d.metadata.get("page", "?"),

                "snippet":
                    d.page_content[:200]
            })

        return {
            "answer": answer,
            "sources": sources
        }

    except Exception as e:

        return {
            "error": str(e)
        }


print("✅ RAG Engine Ready")


# ============================================================
# STEP 5: FASTAPI APP
# ============================================================

app = FastAPI()


class QueryRequest(BaseModel):

    question: str


HTML_UI = """

<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta name="viewport"
content="width=device-width, initial-scale=1.0">

<title>Production RAG Assistant[ Maisha_45]</title>

<style>

*{
margin:0;
padding:0;
box-sizing:border-box;
}

body{
font-family:'Segoe UI',sans-serif;
background:#0f172a;
color:#e2e8f0;
display:flex;
flex-direction:column;
height:100vh;
}

header{
background:#1e293b;
padding:16px 24px;
border-bottom:1px solid #334155;
}

header h1{
font-size:1.2rem;
color:#38bdf8;
}

#upload-bar{
background:#1e293b;
padding:12px 24px;
border-bottom:1px solid #334155;
display:flex;
gap:10px;
align-items:center;
}

#upload-bar input{
flex:1;
color:#94a3b8;
}

button{
background:#0ea5e9;
color:white;
border:none;
padding:10px 18px;
border-radius:8px;
cursor:pointer;
}

button:hover{
background:#0284c7;
}

#status{
font-size:0.85rem;
color:#94a3b8;
}

#chat{
flex:1;
overflow-y:auto;
padding:20px;
display:flex;
flex-direction:column;
gap:12px;
}

.msg{
max-width:78%;
padding:12px 16px;
border-radius:12px;
line-height:1.5;
white-space:pre-wrap;
}

.user{
align-self:flex-end;
background:#0ea5e9;
color:white;
border-bottom-right-radius:4px;
}

.bot{
align-self:flex-start;
background:#1e293b;
border:1px solid #334155;
border-bottom-left-radius:4px;
}

.src{
margin-top:8px;
padding-top:8px;
border-top:1px solid #334155;
font-size:0.75rem;
color:#94a3b8;
}

#input-row{
display:flex;
gap:10px;
padding:16px 24px;
background:#1e293b;
border-top:1px solid #334155;
}

#question{
flex:1;
background:#0f172a;
border:1px solid #334155;
color:white;
padding:10px 14px;
border-radius:8px;
outline:none;
}

#question:focus{
border-color:#0ea5e9;
}

.spinner{
display:inline-block;
width:14px;
height:14px;
border:2px solid #334155;
border-top-color:#38bdf8;
border-radius:50%;
animation:spin .7s linear infinite;
}

@keyframes spin{
to{
transform:rotate(360deg);
}
}

</style>

</head>

<body>

<header>
<h1>🤖 Production-Stable RAG Assistant_[Maisha_74]</h1>
</header>

<div id="upload-bar">

<input
    type="file"
    id="pdfFile"
    accept=".pdf"
/>

<button onclick="uploadPDF()">
📄 Upload PDF
</button>

<span id="status">
No document loaded
</span>

</div>

<div id="chat"></div>

<div id="input-row">

<input
id="question"
placeholder="Ask something about your document..."
onkeydown="if(event.key==='Enter') askQuestion()"
/>

<button onclick="askQuestion()">
Send ➤
</button>

</div>

<script>

const BASE_URL = window.location.origin;

const chat =
document.getElementById("chat");

const statusText =
document.getElementById("status");


function addMessage(
text,
role,
sources=[]
){

const div =
document.createElement("div");

div.className =
"msg " + role;

div.innerText = text;

if(sources.length){

const srcDiv =
document.createElement("div");

srcDiv.className = "src";

srcDiv.innerText =
"📄 Sources: " +

sources.map(
s => "Page " + (s.page + 1)
).join(", ");

div.appendChild(srcDiv);
}

chat.appendChild(div);

chat.scrollTop =
chat.scrollHeight;

return div;
}


async function uploadPDF(){

const file =
document.getElementById(
"pdfFile"
).files[0];

if(!file){
alert(
"Please select a PDF first."
);
return;
}

statusText.innerHTML =
'<span class="spinner"></span> Processing...';

const formData =
new FormData();

formData.append(
"file",
file
);

try{

const response =
await fetch(
BASE_URL + "/upload",
{
method:"POST",
body:formData
}
);

const data =
await response.json();

statusText.textContent =
data.message || data.error;

if(data.chunks){

addMessage(
`📄 Loaded:\n${data.pages} pages\n${data.chunks} chunks\n\nAsk me anything!`,
"bot"
);
}

}catch(err){

statusText.textContent =
"Upload failed.";
}
}


async function askQuestion(){

const q =
document.getElementById(
"question"
);

if(!q.value.trim()) return;

const question = q.value;

addMessage(question, "user");

q.value = "";

const thinking =
addMessage(
"⏳ Thinking...",
"bot"
);

try{

const response =
await fetch(
BASE_URL + "/ask",
{
method:"POST",
headers:{
"Content-Type":"application/json"
},
body:JSON.stringify({
question:question
})
}
);

const data =
await response.json();

chat.removeChild(thinking);

if(data.error){

addMessage(
data.error,
"bot"
);

}else{

addMessage(
data.answer,
"bot",
data.sources || []
);
}

}catch(err){

chat.removeChild(thinking);

addMessage(
"❌ Error: Could not reach server.",
"bot"
);
}
}

</script>

</body>
</html>

"""


# ============================================================
# ROUTES
# ============================================================

@app.get("/", response_class=HTMLResponse)
async def home():

    return HTML_UI


@app.post("/upload")
async def upload_pdf(
    file: UploadFile = File(...)
):

    try:

        destination = (
            UPLOAD_DIR / file.filename
        )

        with open(destination, "wb") as buffer:

            shutil.copyfileobj(
                file.file,
                buffer
            )

        return ingest_pdf(str(destination))

    except Exception as e:

        return {
            "error": str(e)
        }


@app.post("/ask")
async def ask(
    req: QueryRequest
):

    return ask_question(req.question)


print("✅ FastAPI App Ready")


# ============================================================
# STEP 6: START SERVER + NGROK
# ============================================================

PORT = 8000

# Kill previous tunnels
ngrok.kill()

# Create public tunnel
public_url = ngrok.connect(PORT)

print(f"\n🚀 Public URL:\n{public_url}\n")


# Start FastAPI Server

def start_server():

    config = uvicorn.Config(
        app=app,
        host="0.0.0.0",
        port=PORT,
        log_level="info"
    )

    server = uvicorn.Server(config)

    loop = asyncio.new_event_loop()

    asyncio.set_event_loop(loop)

    loop.run_until_complete(
        server.serve()
    )


# Run in background thread
server_thread = threading.Thread(
    target=start_server,
    daemon=True
)

server_thread.start()

print("✅ FastAPI Server Running")


# ============================================================
# OPTIONAL: TEST LLM
# ============================================================

# Uncomment to test

# response = LLM.invoke("Hello")
# print(response.content)

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-77' coro=<Server.serve() done, defined at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:77> exception=SystemExit(1)>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 172, in startup
    server = await loop.create_server(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/base_events.py", line 1584, in create_server
    raise OSError(err.errno, msg) from None
OSError: [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_69939/256048887.py", l

✅ Packages installed successfully
✅ Imports successful
✅ GROQ API Key Loaded


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ RAG Engine Ready
✅ FastAPI App Ready

🚀 Public URL:
NgrokTunnel: "https://mowing-paralyze-oppressed.ngrok-free.dev" -> "http://localhost:8000"

✅ FastAPI Server Running


INFO:     Started server process [69939]
